In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sys
import os

project_root = os.path.dirname(os.path.abspath(''))
if project_root not in sys.path:
    sys.path.append(project_root)

from FEATURES.features import *
from FEATURES.featuresV2 import *
from PRODUCTION.calculateEVS import *
from MODELS.pipeline import *
from MODELS.teamInfo import teamStarPlayer, projectedStartingFive, mainStartingFive

### Load Model

In [2]:
# Load split NGBoost models (mean, variance, and calibration factor)
pts_mean_model = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_MEAN_MODEL_PRODUCTION.pkl')
pts_var_model = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_VAR_MODEL_PRODUCTION.pkl')
calibration_factor = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_CALIBRATION_FACTOR_PRODUCTION.pkl')
model = (pts_mean_model, pts_var_model, calibration_factor)  
features = joblib.load('../MODELS/SAVED_MODELS/feature_list.pkl')

print(f"Loaded models with calibration factor: {calibration_factor}")

Loaded models with calibration factor: 1


### Load Player Data and Bookmaker Data

In [9]:
pd.set_option('display.max_columns', None)
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

s26 = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_26.csv').sort_values(by='GAME_DATE')

usData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_US_{today}.csv')
dfsData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_DFS_{today}.csv')
dfsData.head()

,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE
0,PrizePicks,player_points,Miles Bridges,Over,23.5,-137,2025-11-13,2025-11-12T19:21:18Z
1,PrizePicks,player_points,Miles Bridges,Under,23.5,-137,2025-11-13,2025-11-12T19:21:18Z
2,PrizePicks,player_points,Kon Knueppel,Over,18.5,-137,2025-11-13,2025-11-12T19:21:18Z
3,PrizePicks,player_points,Kon Knueppel,Under,18.5,-137,2025-11-13,2025-11-12T19:21:18Z
4,PrizePicks,player_points,Ryan Rollins,Over,18.5,-137,2025-11-13,2025-11-12T19:21:18Z


### Update projected starting lineups

In [5]:
from MODELS.scrapStarting import NBADailyLineups

scraper = NBADailyLineups("https://www.rotowire.com/basketball/nba-lineups.php")
scraper.getDict()  # Scrape the lineups
scraper.updateTeamInfo()  # Update teamInfo.py

No data available. Run getDict() first.


### Top EVs for single bets

In [ ]:
singlePTSBookies = usData[(usData['CATEGORY'] == 'player_points') & (usData['ODDS'] <= 250) & (usData['ODDS'] >= -250)]

results = calculateSingleBets(s26, singlePTSBookies, model, features, current_date, 
                             edge_threshold=0.20, stake=10, 
                             variance_inflation=1.1, 
                             use_monte_carlo=True, n_simulations=10000, max_kelly=0.25)  


singleBets = results.sort_values(by='EV$', ascending=False).reset_index(drop=True)
singleBets = singleBets[['NAME', 'BOOKMAKER','LINE', 'PREDICTION','SIDE','ODDS','RECOMMENDATION', 'EV$', 'EXPECTED ROI', 'KELLY_FRACTION','SIGMA FLAG']].head(10)
singleBets.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/singleBets.csv', index=False)
singleBets.head(5)

## Top EVs for 2 leg bets

### Underdog picks

In [8]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

results = calculate2LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=0.35, stake=10, 
                     variance_inflation=1.1,
                     use_monte_carlo=True, n_simulations=10000, max_kelly=0.25)

underdogPairs = results.sort_values(by='EV$', ascending=False).reset_index(drop=True)
underdogPairs = underdogPairs[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2', 'MODEL SIDE 1', 'MODEL SIDE 2', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2']].head(10)
underdogPairs.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogPairs.csv', index=False)
underdogPairs.head()

,NAME 1,NAME 2,LINE 1,LINE 2,MODEL SIDE 1,MODEL SIDE 2,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2
0,Stephon Castle,Will Richard,16.5,6.5,over,over,1,6.42,0.321,High,High
1,Will Richard,Jordan Goodwin,6.5,3.5,over,over,1,5.99,0.300,High,Low
2,Landry Shamet,Will Richard,4.5,6.5,over,over,1,5.91,0.296,High,High
3,Stephon Castle,Jordan Goodwin,16.5,3.5,over,over,1,5.86,0.293,High,Low
4,Landry Shamet,Stephon Castle,4.5,16.5,over,over,1,5.86,0.293,High,High


### Prizepicks picks

In [10]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

results = calculate2LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=0.35, stake=10, 
                     variance_inflation=1.1,
                     use_monte_carlo=True, n_simulations=10000, max_kelly=0.25)

pairsPrizepicks = results.sort_values(by='EV$', ascending=False).reset_index(drop=True)
pairsPrizepicks = pairsPrizepicks[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2', 'MODEL SIDE 1', 'MODEL SIDE 2', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2']].head(10)
pairsPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksPairs.csv', index=False)
pairsPrizepicks.head()

,NAME 1,NAME 2,LINE 1,LINE 2,MODEL SIDE 1,MODEL SIDE 2,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2
0,Caris LeVert,Victor Wembanyama,16.5,18.5,under,over,1,12.67,0.633,Med,High
1,Caris LeVert,Jaylen Brown,16.5,20.5,under,over,1,10.40,0.520,Med,High
2,Jaylen Brown,Victor Wembanyama,20.5,18.5,over,over,1,10.38,0.519,High,High
3,Caris LeVert,Kevon Looney,16.5,4.5,under,over,1,9.66,0.483,Med,Med
4,Victor Wembanyama,Kevon Looney,18.5,4.5,over,over,1,9.62,0.481,High,Med


## 3 leg parlay

### Underdog picks

In [ ]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

threeLeg = calculate3LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=0.60, stake=10, 
                     variance_inflation=1.1, 
                     use_monte_carlo=True, n_simulations=10000, max_kelly=0.25)

underdogTrios = threeLeg.sort_values(by='EV$', ascending=False).reset_index(drop=True)
underdogTrios = underdogTrios[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3','MODEL SIDE 1', 'MODEL SIDE 2', 'MODEL SIDE 3', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']]
underdogTrios.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogTrios.csv', index=False)
underdogTrios.head()

Pre-computing predictions for 117 players...
Processing 107 players with valid predictions...
Generated 198292 valid 3-leg combinations


,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,MODEL SIDE 1,MODEL SIDE 2,MODEL SIDE 3,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2,SIGMA FLAG 3
0,Will Richard,Luka Dončić,John Collins,6.5,31.5,16.5,over,under,under,1,15.29,0.306,High,High,Med
1,Stephon Castle,Will Richard,John Collins,16.5,6.5,16.5,over,over,under,1,14.89,0.298,High,High,Med
2,Stephon Castle,Luka Dončić,John Collins,16.5,31.5,16.5,over,under,under,1,14.76,0.295,High,High,Med
3,Jordan Goodwin,Luka Dončić,John Collins,3.5,31.5,16.5,over,under,under,1,14.55,0.291,Low,High,Med
4,Will Richard,Jordan Goodwin,John Collins,6.5,3.5,16.5,over,over,under,1,14.54,0.291,High,Low,Med


### Prizepicks picks

In [12]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

triosPrizepicks = calculate3LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=0.60, stake=100, 
                     variance_inflation=1.1, 
                     use_monte_carlo=False, n_simulations=10000, max_kelly=0.25)

triosPrizepicks = threeLeg.sort_values(by='EV$', ascending=False).reset_index(drop=True)
triosPrizepicks = triosPrizepicks[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'MODEL SIDE 1', 'MODEL SIDE 2', 'MODEL SIDE 3', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']]
triosPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksTrios.csv', index=False)
triosPrizepicks.head()

Pre-computing predictions for 165 players...
Processing 155 players with valid predictions...
Generated 607960 valid 3-leg combinations


,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,MODEL SIDE 1,MODEL SIDE 2,MODEL SIDE 3,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2,SIGMA FLAG 3
0,Will Richard,Luka Dončić,John Collins,6.5,31.5,16.5,over,under,under,1,15.29,0.306,High,High,Med
1,Stephon Castle,Will Richard,John Collins,16.5,6.5,16.5,over,over,under,1,14.89,0.298,High,High,Med
2,Stephon Castle,Luka Dončić,John Collins,16.5,31.5,16.5,over,under,under,1,14.76,0.295,High,High,Med
3,Jordan Goodwin,Luka Dončić,John Collins,3.5,31.5,16.5,over,under,under,1,14.55,0.291,Low,High,Med
4,Will Richard,Jordan Goodwin,John Collins,6.5,3.5,16.5,over,over,under,1,14.54,0.291,High,Low,Med
